# Regular Expressions in Python — Practice Workbook

**Dataset:** Buddhist universities and colleges from Wikipedia  
**How to use:** Work through the exercises in order. Each exercise has a task, hints, a solution, and a short explanation. Try the task before reading its solution.

## Contents

1. Load the text and find a phrase
2. Match complete lines
3. Extract university names
4. Extract city and state
5. Use named groups and `finditer()`
6. Extract additional details from messy entries
7. Rewrite a pattern using `re.VERBOSE`
8. Quick reference

## Setup — load the dataset

A text file is read into Python as one string. The newline character `\n` separates its lines.

The cell below looks for your local `buddhist_universities.txt` file (or `datasets/buddhist_universities.txt`). If neither exists, it uses the text supplied in the conversation so the notebook can still run.

In [1]:
import re
from pathlib import Path

file_options = [
    Path("buddhist_universities.txt"),
    Path("datasets/buddhist_universities.txt"),
]

file_path = next((path for path in file_options if path.is_file()), None)

if file_path:
    wiki = file_path.read_text(encoding="utf-8")
    print(f"Loaded: {file_path}")
else:
    wiki = """There are several Buddhist universities in the United States. Some of these have existed for decades and are accredited. Others are relatively new and are either in the process of being accredited or else have no formal accreditation. The list includes:

Dhammakaya Open University – located in Azusa, California, part of the Thai Wat Phra Dhammakaya
Dharmakirti College – located in Tucson, Arizona Now called Awam Tibetan Buddhist Institute (http://awaminstitute.org/)
Dharma Realm Buddhist University – located in Ukiah, California (Accredited by the WASC Senior College and University Commission)
Ewam Buddhist Institute – located in Arlee, Montana
Naropa University is located in Boulder, Colorado (Accredited by the Higher Learning Commission)
Institute of Buddhist Studies – located in Berkeley, California
Maitripa College – located in Portland, Oregon
Soka University of America – located in Aliso Viejo, California
University of the West – located in Rosemead, California (Accredited by the WASC Senior College and University Commission)
Won Institute of Graduate Studies – located in Glenside, Pennsylvania

== References ==

== External links ==
List of Buddhist Universities and Colleges in the world
"""
    print("Using the text included in this notebook.")

print("Characters:", len(wiki))
print("Lines:", len(wiki.splitlines()))

Using the text included in this notebook.
Characters: 1214
Lines: 17


## A small regex foundation

A regular expression (regex) is a pattern used to search or extract text.

| Syntax | Meaning |
|---|---|
| `.` | Any character except a newline by default |
| `\d` | A digit |
| `\w` | A word character: letters, digits, underscore |
| `\s` | Whitespace, including spaces and tabs |
| `[abc]` | One character: `a`, `b`, or `c` |
| `[^,]` | One character that is not a comma |
| `*` | Zero or more repetitions (greedy by default) |
| `+` | One or more repetitions (greedy by default) |
| `?` | Zero or one repetition; after `*` or `+`, makes it non-greedy |
| `^` / `$` | Start / end of string, or of each line with `re.MULTILINE` |
| `( ... )` | Capturing group |
| `(?: ... )` | Non-capturing group |
| `A\|B` | Match A or B |

Use raw strings such as `r"\d+"` for regex patterns. The `r` prevents Python from interpreting backslashes as string escapes.

## Exercise 1 — Find a phrase

**Goal:** Find every occurrence of the exact phrase `located in`, then locate the first occurrence.

**Try it:**

1. Use `re.findall()` to get all occurrences.
2. Count them with `len()`.
3. Use `re.search()` to get the first match.
4. Print the matched text and its starting index.

**Hint:** `match.group()` returns the matched text; `match.start()` returns its starting index.

In [2]:
# Solution — Exercise 1
pattern = r"located in"

all_matches = re.findall(pattern, wiki)
first_match = re.search(pattern, wiki)

print("All matches:", all_matches)
print("Number of matches:", len(all_matches))

if first_match is not None:
    print("First match:", first_match.group())
    print("Starting index:", first_match.start())

All matches: ['located in', 'located in', 'located in', 'located in', 'located in', 'located in', 'located in', 'located in', 'located in', 'located in']
Number of matches: 10
First match: located in
Starting index: 284


### Explanation

- `re.findall(pattern, wiki)` returns a **list** containing all non-overlapping matches.
- `re.search(pattern, wiki)` returns a **match object** for the first match, or `None` if there is no match.
- `.group()` returns the matched text. `.start()` returns its zero-based starting position.

Expected count for this dataset: **10**.

## Exercise 2 — Match complete university lines

**Goal:** Extract each complete line containing `located in`, rather than extracting only those two words.

**Try it:** Write a pattern that:

1. Starts at the beginning of a line.
2. Contains `located in` somewhere in that line.
3. Continues to the end of the line.
4. Uses `re.findall()` with `re.MULTILINE`.

**Hint:** Combine `^`, `.*`, and `$`.

In [3]:
# Solution — Exercise 2
line_pattern = r"^.*located in.*$"
university_lines = re.findall(line_pattern, wiki, flags=re.MULTILINE)

print("Number of entries:", len(university_lines))
for line in university_lines:
    print(line)

Number of entries: 10
Dhammakaya Open University – located in Azusa, California, part of the Thai Wat Phra Dhammakaya
Dharmakirti College – located in Tucson, Arizona Now called Awam Tibetan Buddhist Institute (http://awaminstitute.org/)
Dharma Realm Buddhist University – located in Ukiah, California (Accredited by the WASC Senior College and University Commission)
Ewam Buddhist Institute – located in Arlee, Montana
Naropa University is located in Boulder, Colorado (Accredited by the Higher Learning Commission)
Institute of Buddhist Studies – located in Berkeley, California
Maitripa College – located in Portland, Oregon
Soka University of America – located in Aliso Viejo, California
University of the West – located in Rosemead, California (Accredited by the WASC Senior College and University Commission)
Won Institute of Graduate Studies – located in Glenside, Pennsylvania


### Explanation

Pattern: `^.*located in.*$`

| Part | Meaning |
|---|---|
| `^` | Beginning of a line when `re.MULTILINE` is enabled |
| `.*` | Any number of non-newline characters before the phrase |
| `located in` | The exact phrase to require |
| `.*` | Any number of non-newline characters after the phrase |
| `$` | End of a line when `re.MULTILINE` is enabled |

`re.MULTILINE` changes how `^` and `$` work. They recognize the beginning and end of **each line**, not just the whole string. It does not make `.` match newlines; `re.DOTALL` does that.

Expected count: **10**.

### Greedy versus non-greedy matching

Quantifiers such as `*` and `+` are **greedy** by default: they try to match as much text as possible while still allowing the complete pattern to match. Adding `?` makes them **non-greedy (lazy)**: they try to match as little as possible while still allowing the complete pattern to match.

Run the example and compare the results.

In [4]:
example = "A123B456B"

print("Greedy:    ", re.findall(r"A.*B", example))
print("Non-greedy:", re.findall(r"A.*?B", example))

Greedy:     ['A123B456B']
Non-greedy: ['A123B']


**Why are the results different?**

- `A.*B`: `.*` expands as far as it can while leaving a `B` for the final `B` in the pattern. It matches `A123B456B`.
- `A.*?B`: `.*?` expands only until the first `B` allows the full pattern to match. It matches `A123B`.

A second example makes this useful for extraction:

- `<b>.*</b>` can match from the first opening tag to the last closing tag.
- `<b>.*?</b>` matches one opening tag through the nearest closing tag that completes the pattern.

Non-greedy does not mean “always stop at the next delimiter”; it means “use the shortest match that lets the entire pattern succeed.”

## Exercise 3 — Extract university names

**Goal:** Extract only the university name from each entry.

Your data has two separator formats:

- `– located in`
- `is located in`

It may also contain different dash characters. The ordinary hyphen `-`, en dash `–`, and em dash `—` are distinct Unicode characters.

**Try it:** Use one regex with a capture group to extract all 10 names. Support the two separator formats and all three dash variants.

**Hints:**

- `(...)` captures text.
- `(?:...)` groups alternatives without capturing them.
- `|` means OR.
- `[-–—]` matches one of the three dash characters.
- `.+?` is a non-greedy capture: it stops expanding once the separator can match.

In [5]:
# Solution — Exercise 3
name_pattern = r"^(.+?)\s+(?:[-–—]\s*located\s+in|is\s+located\s+in)"
university_names = re.findall(name_pattern, wiki, flags=re.MULTILINE)

print(university_names)
print("Number of names:", len(university_names))

['Dhammakaya Open University', 'Dharmakirti College', 'Dharma Realm Buddhist University', 'Ewam Buddhist Institute', 'Naropa University', 'Institute of Buddhist Studies', 'Maitripa College', 'Soka University of America', 'University of the West', 'Won Institute of Graduate Studies']
Number of names: 10


### Explanation

Pattern: `^(.+?)\s+(?:[-–—]\s*located\s+in|is\s+located\s+in)`

| Part | Meaning |
|---|---|
| `^` | Start of each line |
| `(.+?)` | Capture the title, using one or more characters non-greedily |
| `\s+` | One or more whitespace characters before the separator |
| `(?: ... )` | Group the separator alternatives without capturing them |
| `[-–—]` | Match one hyphen-minus, en dash, or em dash |
| `\s*` | Allow zero or more spaces after the dash |
| `located\s+in` | Match `located`, whitespace, then `in` |
| `|` | OR |
| `is\s+located\s+in` | The alternative wording used by Naropa University |

**Why `(.+?)` rather than `(.+)`?** A non-greedy capture stops at the first point where the separator can match. A greedy capture could continue to a later separator if the line contained more than one.

**Why did a pattern using `-` fail?** Your source has an en dash `–`, not the ordinary hyphen `-`. They look similar but are different characters. Inspect them with `repr(char)` and `ord(char)` if needed.

Expected count: **10**.

## Exercise 4 — Extract city and state

**Goal:** Extract three fields from every entry: university name, city, and state.

Example result:

`('Dhammakaya Open University', 'Azusa', 'California')`

**Try it:** Extend Exercise 3 to add two more capture groups. The city comes before the comma; the state comes after it. Ignore any text after the state.

**Hints:**

- `[^,]+` means one or more characters that are not commas.
- `[A-Z][a-z]+` matches a capital letter followed by one or more lowercase letters.
- Use three capturing groups in total.

In [6]:
# Solution — Exercise 4
location_pattern = (
    r"^(.+?)\s+(?:[-–—]\s*located\s+in|is\s+located\s+in)"
    r"\s+([^,]+),\s+([A-Z][a-z]+)"
)

locations = re.findall(location_pattern, wiki, flags=re.MULTILINE)

for location in locations:
    print(location)

print("Number of records:", len(locations))

('Dhammakaya Open University', 'Azusa', 'California')
('Dharmakirti College', 'Tucson', 'Arizona')
('Dharma Realm Buddhist University', 'Ukiah', 'California')
('Ewam Buddhist Institute', 'Arlee', 'Montana')
('Naropa University', 'Boulder', 'Colorado')
('Institute of Buddhist Studies', 'Berkeley', 'California')
('Maitripa College', 'Portland', 'Oregon')
('Soka University of America', 'Aliso Viejo', 'California')
('University of the West', 'Rosemead', 'California')
('Won Institute of Graduate Studies', 'Glenside', 'Pennsylvania')
Number of records: 10


### Explanation

The pattern has three capture groups:

1. `(.+?)` — university name.
2. `([^,]+)` — city: consume characters until the comma.
3. `([A-Z][a-z]+)` — state: one capital letter followed by lowercase letters.

The comma and whitespace between city and state are matched but not captured.

**Dataset limitation:** `[A-Z][a-z]+` assumes a one-word state name made of standard English letters. That works for these ten records, but a more general location parser would need to support names such as `New York` or `West Virginia`.

## Exercise 5 — Named groups and `finditer()`

**Goal:** Produce dictionaries like the lecture screenshot, with keys `title`, `city`, and `state`.

**Try it:**

1. Replace the ordinary capture groups with named groups: `(?P<name>...)`.
2. Use `re.finditer()` to iterate over match objects.
3. Call `.groupdict()` on each match.

**Hint:** Use the same matching logic as Exercise 4; only change the capture-group syntax and iteration method.

In [7]:
# Solution — Exercise 5
named_location_pattern = (
    r"^(?P<title>.+?)\s+(?:[-–—]\s*located\s+in|is\s+located\s+in)"
    r"\s+(?P<city>[^,]+),\s+(?P<state>[A-Z][a-z]+)"
)

for match in re.finditer(named_location_pattern, wiki, flags=re.MULTILINE):
    print(match.groupdict())

{'title': 'Dhammakaya Open University', 'city': 'Azusa', 'state': 'California'}
{'title': 'Dharmakirti College', 'city': 'Tucson', 'state': 'Arizona'}
{'title': 'Dharma Realm Buddhist University', 'city': 'Ukiah', 'state': 'California'}
{'title': 'Ewam Buddhist Institute', 'city': 'Arlee', 'state': 'Montana'}
{'title': 'Naropa University', 'city': 'Boulder', 'state': 'Colorado'}
{'title': 'Institute of Buddhist Studies', 'city': 'Berkeley', 'state': 'California'}
{'title': 'Maitripa College', 'city': 'Portland', 'state': 'Oregon'}
{'title': 'Soka University of America', 'city': 'Aliso Viejo', 'state': 'California'}
{'title': 'University of the West', 'city': 'Rosemead', 'state': 'California'}
{'title': 'Won Institute of Graduate Studies', 'city': 'Glenside', 'state': 'Pennsylvania'}


### Explanation

- `(?P<title>...)` captures a group named `title`.
- `(?P<city>...)` captures a group named `city`.
- `(?P<state>...)` captures a group named `state`.
- `re.finditer()` returns an iterator of match objects. It is useful when you want access to each match's methods and metadata.
- `.groupdict()` converts the named captures into a dictionary.

This is the same general technique used in the lecture screenshot.

## Exercise 6 — Extract additional details

**Goal:** Handle information that appears after the state, such as accreditation, a replacement institution, or a URL.

Examples from the text:

- `(Accredited by the WASC Senior College and University Commission)`
- `Now called Awam Tibetan Buddhist Institute (http://awaminstitute.org/)`
- `, part of the Thai Wat Phra Dhammakaya`

**Try it:**

1. Capture `title`, `city`, `state`, and the remaining text after the state as `extra`.
2. Search `extra` for accreditation information.
3. Search `extra` for a replacement institution and its URL.
4. Keep missing optional fields as `None`.

**Hints:**

- Use `(?P<extra>.*)` to capture the rest of the line.
- `[^)]+` matches characters up to a closing parenthesis.
- `https?://` accepts either `http://` or `https://`.
- Use `re.search()` on `extra` for the optional details.

In [8]:
# Solution — Exercise 6
extra_pattern = (
    r"^(?P<title>.+?)\s+(?:[-–—]\s*located\s+in|is\s+located\s+in)"
    r"\s+(?P<city>[^,]+),\s+(?P<state>[A-Z][a-z]+)(?P<extra>.*)$"
)

accreditation_pattern = r"\(Accredited by the (?P<accreditation>[^)]+)\)"
replacement_pattern = (
    r"\bNow called (?P<replacement>.+?)\s+"
    r"\((?P<url>https?://[^)]+)\)"
)

records = []

for match in re.finditer(extra_pattern, wiki, flags=re.MULTILINE):
    record = match.groupdict()
    extra = record.pop("extra").strip()

    accreditation_match = re.search(accreditation_pattern, extra)
    replacement_match = re.search(replacement_pattern, extra)

    record["extra"] = extra or None
    record["accreditation"] = (
        accreditation_match.group("accreditation")
        if accreditation_match else None
    )
    record["replacement"] = (
        replacement_match.group("replacement").strip()
        if replacement_match else None
    )
    record["url"] = replacement_match.group("url") if replacement_match else None

    records.append(record)

for record in records:
    print(record)

print("Number of records:", len(records))

{'title': 'Dhammakaya Open University', 'city': 'Azusa', 'state': 'California', 'extra': ', part of the Thai Wat Phra Dhammakaya', 'accreditation': None, 'replacement': None, 'url': None}
{'title': 'Dharmakirti College', 'city': 'Tucson', 'state': 'Arizona', 'extra': 'Now called Awam Tibetan Buddhist Institute (http://awaminstitute.org/)', 'accreditation': None, 'replacement': 'Awam Tibetan Buddhist Institute', 'url': 'http://awaminstitute.org/'}
{'title': 'Dharma Realm Buddhist University', 'city': 'Ukiah', 'state': 'California', 'extra': '(Accredited by the WASC Senior College and University Commission)', 'accreditation': 'WASC Senior College and University Commission', 'replacement': None, 'url': None}
{'title': 'Ewam Buddhist Institute', 'city': 'Arlee', 'state': 'Montana', 'extra': None, 'accreditation': None, 'replacement': None, 'url': None}
{'title': 'Naropa University', 'city': 'Boulder', 'state': 'Colorado', 'extra': '(Accredited by the Higher Learning Commission)', 'accredit

### Explanation

The first pattern captures the common fields and stores everything after the state in `extra`:

- `(?P<extra>.*)` captures the rest of the current line.
- `$` ensures the match reaches the end of that line.

Then two smaller patterns inspect that optional text:

- `\(Accredited by the (?P<accreditation>[^)]+)\)` captures the accreditation body inside the parentheses.
- `\bNow called ...` looks for the replacement institution. `(?P<replacement>.+?)` captures its name, and `(?P<url>https?://[^)]+)` captures its HTTP(S) URL.

The `if ... else None` expressions handle entries where no accreditation, replacement name, or URL exists. This is a common data-cleaning pattern: extract the stable structure first, then parse optional fields separately.

## Exercise 7 — Make the regex readable with `re.VERBOSE`

**Goal:** Rewrite the named-group pattern from Exercise 5 in a multiline format with comments.

**Try it:** Use `re.compile()` with `re.VERBOSE` and `re.MULTILINE`. Extract the same `title`, `city`, and `state` fields.

**Hints:**

- In verbose mode, unescaped whitespace outside character classes is ignored.
- Use `\s+` when the pattern needs to match whitespace in the data.
- `#` starts a comment in verbose mode. Use `\#` to match a literal hash character.

In [9]:
# Solution — Exercise 7
verbose_location_pattern = re.compile(r"""
    ^                                               # beginning of a line
    (?P<title>.+?)                                  # university name
    \s+
    (?:                                             # either separator format
        [-–—]\s*located\s+in
        |
        is\s+located\s+in
    )
    \s+
    (?P<city>[^,]+)                                 # city, up to the comma
    ,\s*
    (?P<state>[A-Z][a-z]+)                          # one-word state name
""", flags=re.MULTILINE | re.VERBOSE)

verbose_records = [
    match.groupdict()
    for match in verbose_location_pattern.finditer(wiki)
]

for record in verbose_records:
    print(record)

print("Number of records:", len(verbose_records))

{'title': 'Dhammakaya Open University', 'city': 'Azusa', 'state': 'California'}
{'title': 'Dharmakirti College', 'city': 'Tucson', 'state': 'Arizona'}
{'title': 'Dharma Realm Buddhist University', 'city': 'Ukiah', 'state': 'California'}
{'title': 'Ewam Buddhist Institute', 'city': 'Arlee', 'state': 'Montana'}
{'title': 'Naropa University', 'city': 'Boulder', 'state': 'Colorado'}
{'title': 'Institute of Buddhist Studies', 'city': 'Berkeley', 'state': 'California'}
{'title': 'Maitripa College', 'city': 'Portland', 'state': 'Oregon'}
{'title': 'Soka University of America', 'city': 'Aliso Viejo', 'state': 'California'}
{'title': 'University of the West', 'city': 'Rosemead', 'state': 'California'}
{'title': 'Won Institute of Graduate Studies', 'city': 'Glenside', 'state': 'Pennsylvania'}
Number of records: 10


### Explanation

`re.VERBOSE` lets you spread a regex across multiple lines and add comments. It ignores ordinary whitespace in the pattern, so `\s+` is used wherever the regex must match actual whitespace in the input.

`re.compile()` creates a reusable pattern object. You can call `.finditer()` on that object whenever you need to apply the pattern.

## Quick reference

| Need | Use |
|---|---|
| All matches as a list | `re.findall(pattern, text)` |
| First match object | `re.search(pattern, text)` |
| Iterate through match objects | `re.finditer(pattern, text)` |
| Capture part of a match | `( ... )` |
| Name a capture | `(?P<field>...)` |
| Group without capturing | `(?: ... )` |
| Start / end of each line | `^` / `$` with `re.MULTILINE` |
| Non-greedy repetition | `*?`, `+?`, `??` |
| Dot includes newline | `re.DOTALL` |
| Readable multiline regex | `re.VERBOSE` |

### Key lessons

1. Start with the structure of the text before writing a regex.
2. Inspect punctuation carefully: `-`, `–`, and `—` are different characters.
3. Capture only the fields you need; use non-capturing groups for structural alternatives.
4. Use `re.finditer()` and named groups when producing structured records.
5. For inconsistent text, extract the stable fields first and parse optional details separately.